# Complaint Classification Baseline

Train a majority-class reference and a TF-IDF logistic regression baseline.

Fit models and learned preprocessing on training data only.
Evaluate on the full chronological validation population and the fixed
subset without exact text overlap with the training period.

In [1]:
from pathlib import Path
import json
import time

import pandas as pd
import sklearn

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

PROJECT_ROOT = Path.cwd().parent
assert (PROJECT_ROOT / "PROJECT_PLAN.md").is_file(), (
    "Run this notebook from the project's notebooks directory."
)

processed_dir = PROJECT_ROOT / "data" / "processed"

training = pd.read_parquet(
    processed_dir / "baseline_training.parquet"
)
validation = pd.read_parquet(
    processed_dir / "credit_card_validation_text_flagged.parquet"
)

validation_config = json.loads(
    (PROJECT_ROOT / "configs" / "validation_data.json").read_text(
        encoding="utf-8"
    )
)

assert len(training) == 30180
assert len(validation) == validation_config["full_validation_rows"]
assert training["complaint_id"].is_unique
assert validation["complaint_id"].is_unique
assert training["narrative"].notna().all()
assert validation["narrative"].notna().all()
assert training["issue"].notna().all()
assert validation["issue"].notna().all()

assert set(training["complaint_id"]).isdisjoint(
    set(validation["complaint_id"])
), "Training and validation complaint IDs overlap."

assert pd.to_datetime(training["received_date"]).max() < (
    pd.to_datetime(validation["received_date"]).min()
), "Training and validation dates are not strictly separated."

subset_config = validation_config["comparison_subset"]
no_overlap_mask = validation[subset_config["flag_column"]].eq(
    subset_config["required_value"]
)

assert int(no_overlap_mask.sum()) == subset_config["rows"]

labels = sorted(training["issue"].unique())
assert set(validation["issue"]).issubset(set(labels))

X_train = training["narrative"]
y_train = training["issue"]
X_validation = validation["narrative"]
y_validation = validation["issue"]

display(pd.DataFrame([
    {
        "dataset": "baseline_training",
        "rows": len(training),
        "issue_categories": training["issue"].nunique(),
    },
    {
        "dataset": "full_validation",
        "rows": len(validation),
        "issue_categories": validation["issue"].nunique(),
    },
    {
        "dataset": "validation_no_training_exact_overlap",
        "rows": int(no_overlap_mask.sum()),
        "issue_categories": validation.loc[
            no_overlap_mask, "issue"
        ].nunique(),
    },
]))

print("scikit-learn:", sklearn.__version__)
print("Classification data checks passed.")

,dataset,rows,issue_categories
0,baseline_training,30180,15
1,full_validation,5957,15
2,validation_no_training_exact_overlap,5793,15


scikit-learn: 1.9.1
Classification data checks passed.


In [2]:
def evaluate_predictions(model_name, predictions):
    """Evaluate predictions on both fixed validation populations."""
    rows = []

    for population, mask in [
        ("full_validation", pd.Series(True, index=validation.index)),
        ("no_training_exact_overlap", no_overlap_mask),
    ]:
        actual = y_validation.loc[mask]
        predicted = predictions.loc[mask]

        rows.append({
            "model": model_name,
            "population": population,
            "rows": len(actual),
            "accuracy": accuracy_score(actual, predicted),
            "macro_f1": f1_score(
                actual,
                predicted,
                labels=labels,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                actual,
                predicted,
                labels=labels,
                average="weighted",
                zero_division=0,
            ),
        })

    return pd.DataFrame(rows)


majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train.to_frame(), y_train)

majority_predictions = pd.Series(
    majority_model.predict(X_validation.to_frame()),
    index=validation.index,
    name="majority_prediction",
)

majority_metrics = evaluate_predictions(
    "majority_class",
    majority_predictions,
)

print("Predicted majority class:", majority_predictions.iloc[0])
display(majority_metrics.round(4))

Predicted majority class: Problem with a purchase shown on your statement


,model,population,rows,accuracy,macro_f1,weighted_f1
0,majority_class,full_validation,5957,0.2694,0.0283,0.1144
1,majority_class,no_training_exact_overlap,5793,0.2771,0.0289,0.1202


In [3]:
text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=50000,
            sublinear_tf=True,
            norm="l2",
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            l1_ratio=0.0,
            solver="lbfgs",
            max_iter=1000,
            tol=1e-4,
            class_weight=None,
        ),
    ),
])

# Learn the vocabulary, IDF weights, and classifier from training data only.
fit_start = time.perf_counter()
text_model.fit(X_train, y_train)
text_fit_seconds = time.perf_counter() - fit_start

prediction_start = time.perf_counter()
text_predictions = pd.Series(
    text_model.predict(X_validation),
    index=validation.index,
    name="tfidf_logistic_prediction",
)
text_prediction_seconds = time.perf_counter() - prediction_start

text_metrics = evaluate_predictions(
    "tfidf_logistic",
    text_predictions,
)

baseline_metrics = pd.concat(
    [majority_metrics, text_metrics],
    ignore_index=True,
)

display(baseline_metrics.round(4))

classifier = text_model.named_steps["classifier"]
vectorizer = text_model.named_steps["tfidf"]

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Training seconds:", round(text_fit_seconds, 2))
print("Validation prediction seconds:", round(text_prediction_seconds, 2))
print("Solver iterations:", classifier.n_iter_.tolist())

,model,population,rows,accuracy,macro_f1,weighted_f1
0,majority_class,full_validation,5957,0.2694,0.0283,0.1144
1,majority_class,no_training_exact_overlap,5793,0.2771,0.0289,0.1202
2,tfidf_logistic,full_validation,5957,0.6703,0.4417,0.6591
3,tfidf_logistic,no_training_exact_overlap,5793,0.6613,0.4278,0.6485


Vocabulary size: 50000
Training seconds: 29.82
Validation prediction seconds: 0.74
Solver iterations: [129]


In [4]:
class_report_frames = []

for population, mask in [
    ("full_validation", pd.Series(True, index=validation.index)),
    ("no_training_exact_overlap", no_overlap_mask),
]:
    report = classification_report(
        y_validation.loc[mask],
        text_predictions.loc[mask],
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    class_report = (
        pd.DataFrame(report)
        .T
        .loc[labels, ["precision", "recall", "f1-score", "support"]]
        .rename_axis("issue")
        .reset_index()
        .rename(columns={"f1-score": "f1"})
    )

    class_report["support"] = class_report["support"].astype(int)
    class_report.insert(0, "population", population)
    class_report.insert(0, "model", "tfidf_logistic")

    class_report_frames.append(class_report)

baseline_class_metrics = pd.concat(
    class_report_frames,
    ignore_index=True,
)

print("Per-class results: full validation")
display(
    baseline_class_metrics.loc[
        baseline_class_metrics["population"].eq("full_validation")
    ]
    .sort_values(["f1", "issue"])
    .reset_index(drop=True)
    .round(4)
)

Per-class results: full validation


,model,population,issue,precision,recall,f1,support
0,tfidf_logistic,full_validation,Credit monitoring or identity theft protection...,0.0000,0.0000,0.0000,18
1,tfidf_logistic,full_validation,Problem with fraud alerts or security freezes,0.0000,0.0000,0.0000,6
2,tfidf_logistic,full_validation,Unable to get your credit report or credit score,0.0000,0.0000,0.0000,2
3,tfidf_logistic,full_validation,Improper use of your report,0.5294,0.1698,0.2571,53
4,tfidf_logistic,full_validation,Incorrect information on your report,0.3833,0.3412,0.3610,255
5,tfidf_logistic,full_validation,Struggling to pay your bill,0.6667,0.2647,0.3789,68
6,tfidf_logistic,full_validation,"Other features, terms, or problems",0.4678,0.4785,0.4731,744
7,tfidf_logistic,full_validation,Trouble using your card,0.6237,0.4496,0.5225,258
8,tfidf_logistic,full_validation,Problem when making payments,0.5638,0.4910,0.5249,387
9,tfidf_logistic,full_validation,"Advertising and marketing, including promotion...",0.6878,0.4514,0.5451,288


In [5]:
full_confusion = pd.DataFrame(
    confusion_matrix(
        y_validation,
        text_predictions,
        labels=labels,
    ),
    index=pd.Index(labels, name="actual_issue"),
    columns=pd.Index(labels, name="predicted_issue"),
)

assert int(full_confusion.to_numpy().sum()) == len(validation)

confusion_rows = []

for actual_issue in labels:
    actual_count = int(full_confusion.loc[actual_issue].sum())

    for predicted_issue in labels:
        count = int(full_confusion.loc[actual_issue, predicted_issue])

        if actual_issue != predicted_issue and count > 0:
            confusion_rows.append({
                "actual_issue": actual_issue,
                "predicted_issue": predicted_issue,
                "error_rows": count,
                "actual_class_rows": actual_count,
                "pct_of_actual_class": 100 * count / actual_count,
            })

baseline_confusion_pairs = (
    pd.DataFrame(confusion_rows)
    .sort_values(
        ["error_rows", "actual_issue", "predicted_issue"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print("Most frequent classification errors: full validation")

with pd.option_context("display.max_colwidth", None):
    display(baseline_confusion_pairs.head(15).round(2))

Most frequent classification errors: full validation


,actual_issue,predicted_issue,error_rows,actual_class_rows,pct_of_actual_class
0,"Other features, terms, or problems",Problem with a purchase shown on your statement,132,744,17.74
1,Incorrect information on your report,Getting a credit card,65,255,25.49
2,Problem when making payments,"Other features, terms, or problems",65,387,16.80
3,Problem when making payments,Fees or interest,63,387,16.28
4,Trouble using your card,"Other features, terms, or problems",62,258,24.03
5,"Other features, terms, or problems",Closing your account,58,744,7.80
6,Problem with a purchase shown on your statement,"Other features, terms, or problems",57,1605,3.55
7,Fees or interest,Problem with a purchase shown on your statement,52,701,7.42
8,"Advertising and marketing, including promotional offers","Other features, terms, or problems",50,288,17.36
9,"Advertising and marketing, including promotional offers",Fees or interest,47,288,16.32


In [6]:
prediction_coverage = pd.DataFrame({
    "issue": labels,
    "actual_rows": [
        int(y_validation.eq(label).sum())
        for label in labels
    ],
    "predicted_rows": [
        int(text_predictions.eq(label).sum())
        for label in labels
    ],
    "correct_rows": [
        int(
            (
                y_validation.eq(label)
                & text_predictions.eq(label)
            ).sum()
        )
        for label in labels
    ],
})

print("Classes with no correct predictions")
with pd.option_context("display.max_colwidth", None):
    display(
        prediction_coverage.loc[
            prediction_coverage["correct_rows"].eq(0)
        ].reset_index(drop=True)
    )

Classes with no correct predictions


,issue,actual_rows,predicted_rows,correct_rows
0,Credit monitoring or identity theft protection services,18,0,0
1,Problem with fraud alerts or security freezes,6,0,0
2,Unable to get your credit report or credit score,2,0,0


In [7]:
from sklearn.base import clone

# Change only class weighting; preserve the original fitted baseline.
balanced_model = clone(text_model).set_params(
    classifier__class_weight="balanced"
)

fit_start = time.perf_counter()
balanced_model.fit(X_train, y_train)
balanced_fit_seconds = time.perf_counter() - fit_start

prediction_start = time.perf_counter()
balanced_predictions = pd.Series(
    balanced_model.predict(X_validation),
    index=validation.index,
    name="tfidf_logistic_balanced_prediction",
)
balanced_prediction_seconds = time.perf_counter() - prediction_start

balanced_metrics = evaluate_predictions(
    "tfidf_logistic_balanced",
    balanced_predictions,
)

model_comparison_metrics = pd.concat(
    [majority_metrics, text_metrics, balanced_metrics],
    ignore_index=True,
)

display(model_comparison_metrics.round(4))

print("Balanced training seconds:", round(balanced_fit_seconds, 2))
print(
    "Balanced validation prediction seconds:",
    round(balanced_prediction_seconds, 2),
)
print(
    "Balanced solver iterations:",
    balanced_model.named_steps["classifier"].n_iter_.tolist(),
)

,model,population,rows,accuracy,macro_f1,weighted_f1
0,majority_class,full_validation,5957,0.2694,0.0283,0.1144
1,majority_class,no_training_exact_overlap,5793,0.2771,0.0289,0.1202
2,tfidf_logistic,full_validation,5957,0.6703,0.4417,0.6591
3,tfidf_logistic,no_training_exact_overlap,5793,0.6613,0.4278,0.6485
4,tfidf_logistic_balanced,full_validation,5957,0.6515,0.4480,0.6577
5,tfidf_logistic_balanced,no_training_exact_overlap,5793,0.6420,0.4339,0.6469


Balanced training seconds: 22.06
Balanced validation prediction seconds: 0.73
Balanced solver iterations: [89]


In [8]:
balanced_report = classification_report(
    y_validation,
    balanced_predictions,
    labels=labels,
    output_dict=True,
    zero_division=0,
)

balanced_class_metrics_full = (
    pd.DataFrame(balanced_report)
    .T
    .loc[labels, ["precision", "recall", "f1-score", "support"]]
    .rename_axis("issue")
    .reset_index()
    .rename(columns={"f1-score": "f1"})
)

unweighted_class_metrics_full = baseline_class_metrics.loc[
    baseline_class_metrics["population"].eq("full_validation"),
    ["issue", "precision", "recall", "f1", "support"],
]

class_weight_comparison = unweighted_class_metrics_full.merge(
    balanced_class_metrics_full,
    on="issue",
    suffixes=("_unweighted", "_balanced"),
    validate="one_to_one",
)

class_weight_comparison["f1_change"] = (
    class_weight_comparison["f1_balanced"]
    - class_weight_comparison["f1_unweighted"]
)

with pd.option_context("display.max_colwidth", None):
    display(
        class_weight_comparison[
            [
                "issue",
                "support_unweighted",
                "recall_unweighted",
                "recall_balanced",
                "f1_unweighted",
                "f1_balanced",
                "f1_change",
            ]
        ]
        .sort_values(["f1_change", "issue"], ascending=[False, True])
        .reset_index(drop=True)
        .round(4)
    )

,issue,support_unweighted,recall_unweighted,recall_balanced,f1_unweighted,f1_balanced,f1_change
0,Trouble using your card,258,0.4496,0.7132,0.5225,0.5795,0.0570
1,"Advertising and marketing, including promotional offers",288,0.4514,0.6215,0.5451,0.5977,0.0526
2,Struggling to pay your bill,68,0.2647,0.6029,0.3789,0.4080,0.0290
3,Incorrect information on your report,255,0.3412,0.4863,0.3610,0.3758,0.0148
4,Improper use of your report,53,0.1698,0.4151,0.2571,0.2683,0.0111
5,Problem when making payments,387,0.4910,0.5659,0.5249,0.5355,0.0106
6,Problem with a purchase shown on your statement,1605,0.8835,0.7994,0.8376,0.8410,0.0035
7,Credit monitoring or identity theft protection services,18,0.0000,0.0000,0.0000,0.0000,0.0000
8,Problem with fraud alerts or security freezes,6,0.0000,0.0000,0.0000,0.0000,0.0000
9,Unable to get your credit report or credit score,2,0.0000,0.0000,0.0000,0.0000,0.0000


In [9]:
all_training_text = pd.read_parquet(
    processed_dir / "credit_card_training_text.parquet"
)

assert len(all_training_text) == 33117
assert all_training_text["complaint_id"].is_unique
assert all_training_text["narrative"].notna().all()
assert all_training_text["issue"].notna().all()
assert set(all_training_text["issue"]) == set(labels)

assert set(all_training_text["complaint_id"]).isdisjoint(
    set(validation["complaint_id"])
)

assert pd.to_datetime(all_training_text["received_date"]).max() < (
    pd.to_datetime(validation["received_date"]).min()
)

# Use the original unweighted settings with all training-period text.
all_text_model = clone(text_model)

fit_start = time.perf_counter()
all_text_model.fit(
    all_training_text["narrative"],
    all_training_text["issue"],
)
all_text_fit_seconds = time.perf_counter() - fit_start

prediction_start = time.perf_counter()
all_text_predictions = pd.Series(
    all_text_model.predict(X_validation),
    index=validation.index,
    name="tfidf_logistic_all_training_text_prediction",
)
all_text_prediction_seconds = time.perf_counter() - prediction_start

all_text_metrics = evaluate_predictions(
    "tfidf_logistic_all_training_text",
    all_text_predictions,
)

model_comparison_metrics = pd.concat(
    [
        majority_metrics,
        text_metrics,
        balanced_metrics,
        all_text_metrics,
    ],
    ignore_index=True,
)

display(model_comparison_metrics.round(4))

print("All-text training rows:", len(all_training_text))
print(
    "All-text vocabulary size:",
    len(all_text_model.named_steps["tfidf"].vocabulary_),
)
print("All-text training seconds:", round(all_text_fit_seconds, 2))
print(
    "All-text validation prediction seconds:",
    round(all_text_prediction_seconds, 2),
)
print(
    "All-text solver iterations:",
    all_text_model.named_steps["classifier"].n_iter_.tolist(),
)

,model,population,rows,accuracy,macro_f1,weighted_f1
0,majority_class,full_validation,5957,0.2694,0.0283,0.1144
1,majority_class,no_training_exact_overlap,5793,0.2771,0.0289,0.1202
2,tfidf_logistic,full_validation,5957,0.6703,0.4417,0.6591
3,tfidf_logistic,no_training_exact_overlap,5793,0.6613,0.4278,0.6485
4,tfidf_logistic_balanced,full_validation,5957,0.6515,0.4480,0.6577
5,tfidf_logistic_balanced,no_training_exact_overlap,5793,0.6420,0.4339,0.6469
6,tfidf_logistic_all_training_text,full_validation,5957,0.6706,0.4422,0.6590
7,tfidf_logistic_all_training_text,no_training_exact_overlap,5793,0.6617,0.4288,0.6486


All-text training rows: 33117
All-text vocabulary size: 50000
All-text training seconds: 32.23
All-text validation prediction seconds: 0.76
All-text solver iterations: [139]


In [10]:
text_experiments = [
    ("tfidf_logistic", text_predictions),
    ("tfidf_logistic_balanced", balanced_predictions),
    ("tfidf_logistic_all_training_text", all_text_predictions),
]

classification_report_frames = []

for model_name, predictions in text_experiments:
    for population, mask in [
        ("full_validation", pd.Series(True, index=validation.index)),
        ("no_training_exact_overlap", no_overlap_mask),
    ]:
        report = classification_report(
            y_validation.loc[mask],
            predictions.loc[mask],
            labels=labels,
            output_dict=True,
            zero_division=0,
        )

        frame = (
            pd.DataFrame(report)
            .T
            .loc[labels, ["precision", "recall", "f1-score", "support"]]
            .rename_axis("issue")
            .reset_index()
            .rename(columns={"f1-score": "f1"})
        )

        frame["support"] = frame["support"].astype(int)
        frame.insert(0, "population", population)
        frame.insert(0, "model", model_name)
        classification_report_frames.append(frame)

classification_class_metrics = pd.concat(
    classification_report_frames,
    ignore_index=True,
)

assert len(classification_class_metrics) == 3 * 2 * len(labels)

classification_runtime_summary = pd.DataFrame([
    {
        "model": "tfidf_logistic",
        "training_rows": len(training),
        "fit_seconds": text_fit_seconds,
        "validation_prediction_seconds": text_prediction_seconds,
        "solver_iterations": int(
            text_model.named_steps["classifier"].n_iter_.max()
        ),
    },
    {
        "model": "tfidf_logistic_balanced",
        "training_rows": len(training),
        "fit_seconds": balanced_fit_seconds,
        "validation_prediction_seconds": balanced_prediction_seconds,
        "solver_iterations": int(
            balanced_model.named_steps["classifier"].n_iter_.max()
        ),
    },
    {
        "model": "tfidf_logistic_all_training_text",
        "training_rows": len(all_training_text),
        "fit_seconds": all_text_fit_seconds,
        "validation_prediction_seconds": all_text_prediction_seconds,
        "solver_iterations": int(
            all_text_model.named_steps["classifier"].n_iter_.max()
        ),
    },
])

print("Per-class report rows:", len(classification_class_metrics))
display(classification_runtime_summary.round(2))

Per-class report rows: 90


,model,training_rows,fit_seconds,validation_prediction_seconds,solver_iterations
0,tfidf_logistic,30180,29.82,0.74,129
1,tfidf_logistic_balanced,30180,22.06,0.73,89
2,tfidf_logistic_all_training_text,33117,32.23,0.76,139


In [11]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

report_outputs = [
    ("classification_baseline_metrics", model_comparison_metrics),
    ("classification_class_metrics", classification_class_metrics),
    ("classification_runtime_summary", classification_runtime_summary),
    ("classification_baseline_confusion_pairs", baseline_confusion_pairs),
    ("classification_baseline_prediction_coverage", prediction_coverage),
    ("classification_class_weight_comparison", class_weight_comparison),
]

for name, frame in report_outputs:
    frame.to_csv(reports_dir / f"{name}.csv", index=False)

full_confusion.to_csv(
    reports_dir / "classification_baseline_confusion_matrix.csv"
)

# Preserve aligned predictions locally for later error analysis.
classification_predictions = validation[
    [
        "complaint_id",
        "received_date",
        "issue",
        "exact_overlap_with_baseline",
        "exact_overlap_with_training_period",
        "is_no_training_exact_overlap",
    ]
].copy()

classification_predictions["majority_prediction"] = majority_predictions
classification_predictions["tfidf_logistic_prediction"] = text_predictions
classification_predictions["tfidf_logistic_balanced_prediction"] = (
    balanced_predictions
)
classification_predictions["tfidf_logistic_all_training_text_prediction"] = (
    all_text_predictions
)

assert len(classification_predictions) == len(validation)
assert classification_predictions.filter(
    like="_prediction"
).notna().all().all()

predictions_path = (
    processed_dir / "classification_baseline_validation_predictions.parquet"
)
classification_predictions.to_parquet(predictions_path, index=False)

pd.testing.assert_frame_equal(
    pd.read_parquet(predictions_path),
    classification_predictions,
)

print("Seven classification summary reports saved.")
print("Local validation predictions saved and verified.")

Seven classification summary reports saved.
Local validation predictions saved and verified.


In [12]:
import hashlib
import joblib

model_dir = PROJECT_ROOT / "artifacts" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

fitted_models = {
    "tfidf_logistic": text_model,
    "tfidf_logistic_balanced": balanced_model,
    "tfidf_logistic_all_training_text": all_text_model,
}

expected_predictions = {
    name: predictions
    for name, predictions in text_experiments
}

model_paths = {}

for name, model in fitted_models.items():
    path = model_dir / f"{name}.joblib"
    joblib.dump(model, path, compress=3)

    restored_model = joblib.load(path)
    restored_predictions = pd.Series(
        restored_model.predict(X_validation),
        index=validation.index,
    )

    assert restored_predictions.equals(
        expected_predictions[name].rename(None)
    ), f"Reloaded predictions differ for {name}."

    model_paths[name] = path
    print("Model saved and reload verified:", name)

Model saved and reload verified: tfidf_logistic
Model saved and reload verified: tfidf_logistic_balanced
Model saved and reload verified: tfidf_logistic_all_training_text


In [13]:
def classification_file_metadata(path):
    """Record the file used by an experiment."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return {
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": digest.hexdigest(),
    }


experiment_configs = []

for name, model in fitted_models.items():
    vectorizer_params = model.named_steps["tfidf"].get_params()
    classifier_params = model.named_steps["classifier"].get_params()

    input_path = processed_dir / (
        "credit_card_training_text.parquet"
        if name == "tfidf_logistic_all_training_text"
        else "baseline_training.parquet"
    )

    experiment_configs.append({
        "model": name,
        "training_file": classification_file_metadata(input_path),
        "tfidf": {
            key: vectorizer_params[key]
            for key in [
                "lowercase", "analyzer", "ngram_range", "min_df",
                "max_df", "max_features", "sublinear_tf", "norm",
                "use_idf", "smooth_idf", "stop_words", "token_pattern",
            ]
        },
        "classifier": {
            key: classifier_params[key]
            for key in [
                "C", "l1_ratio", "solver", "max_iter",
                "tol", "class_weight", "fit_intercept",
            ]
        },
        "local_model_file": classification_file_metadata(model_paths[name]),
    })

classification_config = {
    "experiment_version": 1,
    "scikit_learn_version": sklearn.__version__,
    "model_input": "narrative",
    "target": "issue",
    "labels": labels,
    "reference_model": {"strategy": "most_frequent"},
    "validation_file": classification_file_metadata(
        processed_dir / "credit_card_validation_text_flagged.parquet"
    ),
    "validation_policy_file": classification_file_metadata(
        PROJECT_ROOT / "configs" / "validation_data.json"
    ),
    "metric_labels": "All 15 training labels",
    "zero_division": 0,
    "fit_preprocessing_on_training_only": True,
    "final_test_evaluated": False,
    "final_model_selected": False,
    "experiments": experiment_configs,
}

config_dir = PROJECT_ROOT / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

_ = (config_dir / "classification_baseline.json").write_text(
    json.dumps(classification_config, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Classification experiment configuration saved.")

Classification experiment configuration saved.


In [16]:
metrics_table = model_comparison_metrics.to_csv(
    index=False,
    float_format="%.4f",
)

indented_metrics = "\n".join(
    "    " + line
    for line in metrics_table.strip().splitlines()
)

classification_report_text = f"""# Complaint Classification Baseline

## Design

- Training period: April 2024 through February 2025.
- Validation period: March through April 2025.
- Input: complaint narrative only.
- Target: 15 issue categories.
- Cleaned training sample: 30,180 records.
- Full training-period text sample: 33,117 records.
- Full validation population: 5,957 records.
- Fixed secondary population: 5,793 records without exact normalized
  text overlap with any training-period narrative.
- Vocabulary and IDF weights are fitted on each experiment's training data.
- Final-test data has not been evaluated.

## Experiments

1. Majority-class reference fitted on the cleaned training sample.
2. TF-IDF logistic regression fitted on the cleaned sample.
3. The same model with balanced class weights.
4. Unweighted TF-IDF logistic regression fitted on all training-period text.

TF-IDF uses word unigrams and bigrams with up to 50,000 features.
Logistic regression uses L2 regularization, C=1.0, and the lbfgs solver.

## Validation results

The following results are in CSV format:

{indented_metrics}

## Findings

- Text models substantially outperform the majority-class reference.
- Balanced weighting raises macro-F1 slightly while reducing accuracy.
  Improvements vary by class; it is not uniformly better.
- The three rarest validation classes have no correct predictions under
  either cleaned-sample weighting setting.
- Cleaned and full-text unweighted models have very similar validation
  results. This experiment does not demonstrate a performance improvement
  from cleaning.
- Cleaning reduces the training sample by approximately 8.9%.
- The sample-policy comparison evaluates conflict exclusion and exact
  deduplication together, not their separate effects.
- Differences between cleaning policies have not been tested for
  statistical significance.

## Limitations and next steps

- These are development results used for model comparison.
- Rare classes have limited validation support.
- Original complaint labels may be ambiguous or inconsistent.
- The secondary population excludes exact overlap; near duplicates and
  shared templates may remain.
- Recorded timings are local batch measurements, not serving benchmarks.
- Classification probabilities have not been assessed for calibration.
- No final model or automatic-routing threshold has been selected.
- Further work includes error review, calibration, and decision-rule
  evaluation before a frozen final-test assessment.
"""

_ = (reports_dir / "classification_baseline.md").write_text(
    classification_report_text,
    encoding="utf-8",
)

print("Classification baseline report saved.")

Classification baseline report saved.
